# Load MSD SQLite database

DB path: `data/MSD_with_all_features.db` (12 GB, 8 tables)

Strategy: keep a persistent `conn`, load smaller feature tables fully into DataFrames, query the 1M-row `merged_partition1` by chunks or sample.

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

DB_PATH = Path('data/MSD_with_all_features.db')
conn = sqlite3.connect(DB_PATH)

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
for t in tables['name']:
    n = pd.read_sql(f'SELECT COUNT(*) AS n FROM "{t}"', conn).iloc[0, 0]
    print(f'{t:30s}  {n:>10,} rows')

## Load metadata + smaller feature tables fully
These are all under 1M rows × 27 columns, so a full pandas load is fine.

In [ ]:
songs               = pd.read_sql('SELECT * FROM songs', conn)
area_of_moments     = pd.read_sql('SELECT * FROM area_of_moments', conn)
lpc                 = pd.read_sql('SELECT * FROM linear_predictive_coding', conn)
low_level           = pd.read_sql('SELECT * FROM low_level_features', conn)
mfcc                = pd.read_sql('SELECT * FROM mfcc_features', conn)
method_of_moments   = pd.read_sql('SELECT * FROM method_of_moments', conn)

for name, df in [('songs', songs), ('area_of_moments', area_of_moments),
                 ('lpc', lpc), ('low_level', low_level),
                 ('mfcc', mfcc), ('method_of_moments', method_of_moments)]:
    print(f'{name:20s}  shape={df.shape}  mem={df.memory_usage(deep=True).sum()/1e6:.1f} MB')

## Cast `songs` columns to proper types
All columns in `songs` are stored as TEXT; convert numeric ones.

In [ ]:
num_cols = ['duration', 'artist_familiarity', 'artist_hotttnesss',
            'year', 'track_7digitalid', 'shs_perf', 'shs_work']
for c in num_cols:
    songs[c] = pd.to_numeric(songs[c], errors='coerce')

songs.dtypes

## The big merged table — load by chunks or sample
`merged_partition1` is 1M rows × 232 columns (~2 GB in memory). Don't load fully unless needed.

Below: load a 10k random sample, plus a chunked iterator for full passes.

In [ ]:
merged_sample = pd.read_sql(
    'SELECT * FROM merged_partition1 ORDER BY RANDOM() LIMIT 10000',
    conn,
)
print('sample shape:', merged_sample.shape)
merged_sample.head(3)

In [ ]:
# Iterate the full merged table in chunks (lazy — change CHUNK to taste)
CHUNK = 50_000
for i, chunk in enumerate(pd.read_sql('SELECT * FROM merged_partition1', conn, chunksize=CHUNK)):
    # process chunk here
    print(f'chunk {i}: {chunk.shape}')
    if i >= 1:  # demo: stop after 2 chunks
        break

## To load the full merged table (uncomment if you really need it)
Estimated memory: ~2 GB. Make sure you have headroom.

In [ ]:
# merged = pd.read_sql('SELECT * FROM merged_partition1', conn)
# print(merged.shape, merged.memory_usage(deep=True).sum()/1e9, 'GB')